In [ ]:
import numpy as np
import pandas as pd
import pickle
import warnings
warnings.filterwarnings('ignore')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from pgmpy.estimators import HillClimbSearch, TreeSearch
from pgmpy.estimators import MaximumLikelihoodEstimator
from pgmpy.structure_score import BIC, K2
from pgmpy.models import BayesianNetwork
from pgmpy.inference import VariableElimination

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.model_selection import StratifiedKFold

import pgmpy
print("pgmpy version:", pgmpy.__version__)
print("All imports OK")

pgmpy version: 1.1.2
All imports OK


In [ ]:
DATA_PATH = "/Users/simonaristovska/Desktop/bachelor thesis/data/preprocessing_objects.pkl"

with open(DATA_PATH, 'rb') as f:
    obj = pickle.load(f)

for k, v in obj.items():
    print(f"{k}: {type(v).__name__}", end="")
    if hasattr(v, 'shape'):
        print(f"  {v.shape}")
    elif hasattr(v, '__len__'):
        print(f"  len={len(v)}")
    else:
        print()

X_train_combined = obj['X_train_combined']
X_test_combined  = obj['X_test_combined']
event_train      = obj['event_train']
event_test       = obj['event_test']
time_train       = obj['time_train']
time_test        = obj['event_test']

print("\n--- Combined columns ---")
for col in sorted(X_train_combined.columns):
    print(col)


df_model: DataFrame  (1902, 687)
df_encoded: DataFrame  (1902, 694)
df_stage: DataFrame  (1402, 688)
target_cols: list  len=4
time_col: str  len=23
event_col: str  len=5
clinical_cols: list  len=20
mutation_cols: list  len=173
mutation_cols_filtered: list  len=115
rare_mutation_cols: list  len=58
expression_cols: list  len=490
top_100_genes: list  len=100
clinical_features: list  len=23
genomic_features: list  len=608
combined_features: list  len=631
genomic_features_filtered: list  len=215
combined_features_filtered: list  len=238
X_train_clinical: DataFrame  (1426, 23)
X_test_clinical: DataFrame  (476, 23)
X_train_genomic: DataFrame  (1426, 215)
X_test_genomic: DataFrame  (476, 215)
X_train_combined: DataFrame  (1426, 238)
X_test_combined: DataFrame  (476, 238)
X_train_combined_scaled: DataFrame  (1426, 238)
X_test_combined_scaled: DataFrame  (476, 238)
time_train: Series  (1426,)
time_test: Series  (476,)
event_train: Series  (1426,)
event_test: Series  (476,)
scaler: StandardScaler

In [ ]:
# вariable selection and BN dataset preparation

# сelected variables (cross-model consensus: SHAP + Lasso Cox + RSF)

CLINICAL_VARS = [
    'age_at_diagnosis',
    'tumor_size',
    'lymph_nodes_examined_positive',
    'neoplasm_histologic_grade',
    'er_status_Positive',
    'pr_status_Positive',
    'her2_status_Positive',
    'chemotherapy',
    'hormone_therapy',
    'cellularity',
    'mutation_count',
    'type_of_breast_surgery_MASTECTOMY',
    '3-gene_classifier_subtype_ER+/HER2- Low Prolif',
]

EXPRESSION_VARS = [   # from SHAP + Lasso Cox genomic
    'stat5a',
    'mmp15',
    'mmp1',
    'igf1',
    'tbx3',
    'hes6',
    'tubb4b',
    'fancd2',
    'ugt2b17',
]

MUTATION_VARS = [     # binary already
    'tp53_mut',
    'gata3_mut',
    'pik3ca_mut',
]

OUTCOME = 'event'

ALL_VARS = CLINICAL_VARS + EXPRESSION_VARS + MUTATION_VARS

# build train / test DataFrames 

df_train = X_train_combined[ALL_VARS].copy()
df_train[OUTCOME] = event_train.values

df_test  = X_test_combined[ALL_VARS].copy()
df_test[OUTCOME]  = event_test.values

print(f"Train: {df_train.shape}  |  Test: {df_test.shape}")
print(f"Total variables: {len(ALL_VARS)} + 1 outcome = {len(ALL_VARS)+1}")
print(f"\nEvent rate — train: {event_train.mean():.3f}  test: {event_test.mean():.3f}")
print(f"\nMissing values:\n{df_train.isnull().sum()[df_train.isnull().sum()>0]}")

# variable types 

BINARY_VARS = [
    'er_status_Positive', 'pr_status_Positive', 'her2_status_Positive',
    'chemotherapy', 'hormone_therapy', 'type_of_breast_surgery_MASTECTOMY',
    '3-gene_classifier_subtype_ER+/HER2- Low Prolif',
    'tp53_mut', 'gata3_mut', 'pik3ca_mut',
]
CONTINUOUS_VARS = [v for v in ALL_VARS if v not in BINARY_VARS]

print(f"\nBinary vars ({len(BINARY_VARS)}): already discrete")
print(f"Continuous vars ({len(CONTINUOUS_VARS)}): need discretization")
print(f"\nContinuous vars:")
for v in CONTINUOUS_VARS:
    print(f"  {v}:  min={df_train[v].min():.2f}  max={df_train[v].max():.2f}")


Train: (1426, 26)  |  Test: (476, 26)
Total variables: 25 + 1 outcome = 26

Event rate — train: 0.327  test: 0.328

Missing values:
Series([], dtype: int64)

Binary vars (10): already discrete
Continuous vars (15): need discretization

Continuous vars:
  age_at_diagnosis:  min=21.93  max=96.29
  tumor_size:  min=1.00  max=180.00
  lymph_nodes_examined_positive:  min=0.00  max=45.00
  neoplasm_histologic_grade:  min=1.00  max=3.00
  cellularity:  min=0.00  max=2.00
  mutation_count:  min=1.00  max=80.00
  stat5a:  min=-3.24  max=4.75
  mmp15:  min=-1.75  max=5.34
  mmp1:  min=-1.38  max=6.28
  igf1:  min=-1.59  max=7.47
  tbx3:  min=-2.14  max=4.72
  hes6:  min=-2.20  max=4.91
  tubb4b:  min=-3.67  max=5.27
  fancd2:  min=-2.94  max=4.62
  ugt2b17:  min=-2.07  max=12.64


In [ ]:
# discretization

# neoplasm_histologic_grade (1-3) and cellularity (0-2) are already ordinal integers
ALREADY_DISCRETE = ['neoplasm_histologic_grade', 'cellularity']
TO_DISCRETIZE    = [v for v in CONTINUOUS_VARS if v not in ALREADY_DISCRETE]

def discretize(df_tr, df_te, strategy='kmeans', n_bins=3):
    """Fit discretizer on train, apply to both. Returns copies."""
    tr = df_tr.copy()
    te = df_te.copy()

    # k-means / quantile on continuous vars
    kbd = KBinsDiscretizer(n_bins=n_bins, encode='ordinal',
                           strategy=strategy, subsample=None)
    tr[TO_DISCRETIZE] = kbd.fit_transform(df_tr[TO_DISCRETIZE]).astype(int)
    te[TO_DISCRETIZE] = kbd.transform(df_te[TO_DISCRETIZE]).astype(int)

    # shift already-discrete ordinals to 0-indexed
    for v in ALREADY_DISCRETE:
        offset = int(df_tr[v].min())
        tr[v] = (df_tr[v] - offset).astype(int)
        te[v] = (df_te[v] - offset).astype(int)

    # cast binary + outcome to int
    for v in BINARY_VARS + [OUTCOME]:
        tr[v] = tr[v].astype(int)
        te[v] = te[v].astype(int)

    return tr, te, kbd

# build both variants
train_km,  test_km,  kbd_km  = discretize(df_train, df_test, strategy='kmeans')
train_qnt, test_qnt, kbd_qnt = discretize(df_train, df_test, strategy='quantile')

print("=== k-means discretization ===")
print(train_km[CONTINUOUS_VARS].head(3).to_string())

print("\n=== quantile discretization ===")
print(train_qnt[CONTINUOUS_VARS].head(3).to_string())

# verify every column is in {0,1,2}
print("\n--- Value check (k-means train) ---")
for col in train_km.columns:
    vals = sorted(train_km[col].unique())
    flag = "OK" if max(vals) <= 2 else "CHECK"
    print(f"  {col}: {vals}  {flag}")

print("\n--- Value check (k-means TEST) ---")
for col in test_km.columns:
    vals = sorted(test_km[col].unique())
    flag = "OK" if max(vals) <= 2 else "CHECK"
    print(f"  {col}: {vals}  {flag}")


=== k-means discretization ===
      age_at_diagnosis  tumor_size  lymph_nodes_examined_positive  neoplasm_histologic_grade  cellularity  mutation_count  stat5a  mmp15  mmp1  igf1  tbx3  hes6  tubb4b  fancd2  ugt2b17
444                  1           0                              0                          1            2               0       1      0     0     1     2     0       0       0        0
964                  0           1                              0                          2            0               1       0      0     0     0     2     0       2       1        0
1751                 2           0                              0                          2            1               0       2      1     1     0     0     1       1       2        0

=== quantile discretization ===
      age_at_diagnosis  tumor_size  lymph_nodes_examined_positive  neoplasm_histologic_grade  cellularity  mutation_count  stat5a  mmp15  mmp1  igf1  tbx3  hes6  tubb4b  fancd2  ugt2b17
444   

In [ ]:
# following the paper: algorithm comparison: 3 algorithms × 2 discretization methods (5-fold CV, AUC)
import time
from pgmpy.causal_discovery import HillClimbSearch as HCS, ExpertKnowledge
from pgmpy.estimators import TreeSearch
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.structure_score import BIC

def to_str(df): return df.astype(str)

BLACKLIST = [(OUTCOME, v) for v in ALL_VARS]
EK = ExpertKnowledge(forbidden_edges=BLACKLIST)

def learn_structure(data_str, algo):
    if algo in ('GHC', 'Tabu'):
        hc  = HCS(scoring_method='bic-d', max_indegree=3,
                  tabu_length=0 if algo == 'GHC' else 10,
                  expert_knowledge=EK, return_type='dag', show_progress=False)
        dag = hc.fit(data_str).causal_graph_
    elif algo == 'TAN':
        ts  = TreeSearch(data_str, root_node='lymph_nodes_examined_positive')
        dag = ts.estimate(estimator_type='tan', class_node=OUTCOME, show_progress=False)
    return dag

def fit_bn(data_str, dag):
    bn = DiscreteBayesianNetwork(dag.edges())
    for node in data_str.columns:
        if node not in bn.nodes():
            bn.add_node(node)
    bn.fit(data_str)
    return bn

def get_auc(bn, X_val_str, y_val):
    probs = bn.predict_probability(X_val_str)
    p1    = probs[f'{OUTCOME}_1'].values
    return roc_auc_score(y_val, p1)

ALGORITHMS = ['GHC', 'Tabu', 'TAN']
DISC_DATA  = {'kmeans': train_km, 'quantile': train_qnt}
y_arr      = df_train[OUTCOME].values
skf        = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print(f"{'Config':<35}  {'Fold AUCs':>35}  {'Mean':>6}  {'Std':>5}  {'Time':>6}")
print("─" * 95)

for disc_name, train_disc in DISC_DATA.items():
    for algo in ALGORITHMS:
        key    = f"{algo} + {disc_name}"
        aucs   = []
        t0     = time.time()

        for fold, (tr_idx, val_idx) in enumerate(skf.split(train_disc, y_arr)):
            tr_f   = to_str(train_disc.iloc[tr_idx].reset_index(drop=True))
            val_f  = to_str(train_disc.iloc[val_idx].reset_index(drop=True))
            X_val  = val_f.drop(columns=[OUTCOME])
            y_val  = train_disc.iloc[val_idx][OUTCOME].values
            try:
                dag = learn_structure(tr_f, algo)
                bn  = fit_bn(tr_f, dag)
                aucs.append(get_auc(bn, X_val, y_val))
            except Exception as e:
                print(f"  [{key} fold {fold}] ERROR: {e}")
                aucs.append(np.nan)

        mean_a = np.nanmean(aucs)
        std_a  = np.nanstd(aucs)
        cv_results[key] = dict(mean_auc=mean_a, std_auc=std_a, fold_aucs=aucs)
        fold_str = "  ".join(f"{a:.4f}" for a in aucs)
        print(f"{key:<35}  [{fold_str}]  {mean_a:.4f}  {std_a:.4f}  ({time.time()-t0:.0f}s)")

print("\n=== RANKING ===")
df_cv = (pd.DataFrame([{'Config': k, 'Mean AUC': v['mean_auc'], 'Std': v['std_auc']}
                        for k, v in cv_results.items()])
          .sort_values('Mean AUC', ascending=False).reset_index(drop=True))
print(df_cv.to_string(index=False))

BEST_CONFIG = df_cv.iloc[0]['Config']
BEST_ALGO, BEST_DISC = BEST_CONFIG.split(' + ')
print(f"\nWinner: {BEST_ALGO} + {BEST_DISC}")


Fitting final BN: TAN + quantile on full training set...
Network: 26 nodes, 49 edges

Test set results:
  AUC : 0.6446
  F1  : 0.4399
  AUC 95% CI: (0.5887 – 0.6976)

Final result: AUC = 0.645 (95% CI: 0.589–0.698), F1 = 0.440


In [ ]:
# final model: TAN + quantile, fit on full train, evaluate on test

BEST_ALGO = 'TAN'
BEST_DISC = 'quantile'

train_final = to_str(train_qnt)
test_final  = to_str(test_qnt)

# fit final BN 

print(f"Fitting final BN: {BEST_ALGO} + {BEST_DISC} on full training set...")
ts_final  = TreeSearch(train_final, root_node='lymph_nodes_examined_positive')
dag_final = ts_final.estimate(estimator_type='tan', class_node=OUTCOME,
                               show_progress=False)
bn_final  = DiscreteBayesianNetwork(dag_final.edges())
for node in train_final.columns:
    if node not in bn_final.nodes():
        bn_final.add_node(node)
bn_final.fit(train_final)
print(f"Network: {len(dag_final.nodes())} nodes, {len(dag_final.edges())} edges")

# ── Test set evaluation ───────────────────────────────────────────────────────
X_test_str = test_final.drop(columns=[OUTCOME])
y_test     = test_qnt[OUTCOME].values

probs_test = bn_final.predict_probability(X_test_str)
p1_test    = probs_test[f'{OUTCOME}_1'].values
preds_test = (p1_test >= 0.5).astype(int)

auc_test = roc_auc_score(y_test, p1_test)
f1_test  = f1_score(y_test, preds_test)

print(f"\nTest set results:")
print(f"  AUC : {auc_test:.4f}")
print(f"  F1  : {f1_test:.4f}")

# bootstrap CI for AUC 

np.random.seed(42)
boot_aucs = []
n = len(y_test)
for _ in range(1000):
    idx  = np.random.choice(n, n, replace=True)
    boot_aucs.append(roc_auc_score(y_test[idx], p1_test[idx]))

ci_lo, ci_hi = np.percentile(boot_aucs, [2.5, 97.5])
print(f"  AUC 95% CI: ({ci_lo:.4f} – {ci_hi:.4f})")
print(f"\nFinal result: AUC = {auc_test:.3f} (95% CI: {ci_lo:.3f}–{ci_hi:.3f}), F1 = {f1_test:.3f}")


Fitting final BN: TAN + quantile on full training set...
Network: 26 nodes, 49 edges

Test set results:
  AUC : 0.6446
  F1  : 0.4399
  AUC 95% CI: (0.5887 – 0.6976)

Final result: AUC = 0.645 (95% CI: 0.589–0.698), F1 = 0.440


In [ ]:
# network visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx

FIG_PATH = "/Users/simonaristovska/Desktop/bachelor thesis/notebooks/"

# node color groups 
COLOR_MAP = {
    'outcome':    '#E74C3C',   # red
    'clinical':   '#2980B9',   # blue
    'expression': '#1A7A6E',   # teal
    'mutation':   '#E67E00',   # orange
}

def node_color(n):
    if n == OUTCOME:              return COLOR_MAP['outcome']
    if n in MUTATION_VARS:        return COLOR_MAP['mutation']
    if n in EXPRESSION_VARS:      return COLOR_MAP['expression']
    return COLOR_MAP['clinical']

G = dag_final

# layout 
event_pos  = {OUTCOME: (0, 1)}
other_nodes = [n for n in G.nodes() if n != OUTCOME]
n_other     = len(other_nodes)
shell_pos  = {n: (np.cos(2*np.pi*i/n_other), -0.2 + 0.6*np.sin(2*np.pi*i/n_other))
              for i, n in enumerate(other_nodes)}
pos = {**event_pos, **shell_pos}

LABELS = {
    'lymph_nodes_examined_positive':              'lymph nodes+',
    '3-gene_classifier_subtype_ER+/HER2- Low Prolif': '3-gene\nER+/HER2- LP',
    'type_of_breast_surgery_MASTECTOMY':          'mastectomy',
    'neoplasm_histologic_grade':                  'grade',
    'er_status_Positive':                         'ER+',
    'pr_status_Positive':                         'PR+',
    'her2_status_Positive':                       'HER2+',
    'age_at_diagnosis':                           'age',
    'tumor_size':                                 'tumor size',
    'mutation_count':                             'mut count',
    'hormone_therapy':                            'hormone Rx',
    'chemotherapy':                               'chemo',
    'cellularity':                                'cellularity',
    'event':                                      'DEATH',
}
labels = {n: LABELS.get(n, n) for n in G.nodes()}

node_colors = [node_color(n) for n in G.nodes()]

fig, ax = plt.subplots(figsize=(16, 12))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=800, alpha=0.92, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels,
                        font_size=6.5, font_color='white',
                        font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#555555',
                       arrows=True, arrowsize=12,
                       connectionstyle='arc3,rad=0.1',
                       width=0.8, alpha=0.7, ax=ax)

legend_patches = [
    mpatches.Patch(color=COLOR_MAP['outcome'],    label='Outcome (death)'),
    mpatches.Patch(color=COLOR_MAP['clinical'],   label='Clinical'),
    mpatches.Patch(color=COLOR_MAP['expression'], label='Gene expression'),
    mpatches.Patch(color=COLOR_MAP['mutation'],   label='Mutation'),
]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10,
          framealpha=0.9)
ax.set_title('Bayesian Network Structure — TAN (quantile discretization)\n'
             f'26 nodes · 49 edges · Test AUC = {auc_test:.3f}',
             fontsize=13, fontweight='bold', pad=15)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_PATH + 'bn_network_structure.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: bn_network_structure.png")


print(f"\nEdges FROM event node ({OUTCOME}):")
for u, v in dag_final.edges():
    if u == OUTCOME: print(f"  {OUTCOME} → {v}")
print(f"\nEdges TO event node:")
for u, v in dag_final.edges():
    if v == OUTCOME: print(f"  {u} → {OUTCOME}")


Saved: bn_network_structure.png

Edges FROM event node (event):
  event → age_at_diagnosis
  event → tumor_size
  event → lymph_nodes_examined_positive
  event → neoplasm_histologic_grade
  event → er_status_Positive
  event → pr_status_Positive
  event → her2_status_Positive
  event → chemotherapy
  event → hormone_therapy
  event → cellularity
  event → mutation_count
  event → type_of_breast_surgery_MASTECTOMY
  event → 3-gene_classifier_subtype_ER+/HER2- Low Prolif
  event → stat5a
  event → mmp15
  event → mmp1
  event → igf1
  event → tbx3
  event → hes6
  event → tubb4b
  event → fancd2
  event → ugt2b17
  event → tp53_mut
  event → gata3_mut
  event → pik3ca_mut

Edges TO event node:


In [ ]:
# systematic experiments (variable subsets + discretization strategies)
import time

SHAP_TOP10 = [
    'lymph_nodes_examined_positive', 'stat5a', 'tumor_size', 'tp53_mut',
    'mmp15', 'mmp1', 'tbx3', 'igf1', 'her2_status_Positive', 'ugt2b17',
]

# prepare discretized datasets

# quantile-4 bins
tr_q4, te_q4, _ = discretize(df_train, df_test, strategy='quantile', n_bins=4)

# domain bins (clinical knowledge cutoffs)
def domain_bins(df_tr, df_te):
    mc_kbd = KBinsDiscretizer(n_bins=3, encode='ordinal',
                              strategy='quantile', subsample=None)
    def apply(df, fit=False):
        d = df.copy()
        d['age_at_diagnosis'] = pd.cut(
            df['age_at_diagnosis'], bins=[-np.inf, 50, 65, np.inf],
            labels=[0,1,2]).astype(int)
        d['tumor_size'] = pd.cut(
            df['tumor_size'], bins=[-np.inf, 20, 50, np.inf],
            labels=[0,1,2]).astype(int)
        d['lymph_nodes_examined_positive'] = pd.cut(
            df['lymph_nodes_examined_positive'],
            bins=[-np.inf, 0.5, 3.5, 9.5, np.inf],
            labels=[0,1,2,3]).astype(int)
        d['neoplasm_histologic_grade'] = (df['neoplasm_histologic_grade'] - 1).astype(int)
        d['cellularity'] = df['cellularity'].astype(int)
        mc = mc_kbd.fit_transform(df[['mutation_count']]) if fit \
             else mc_kbd.transform(df[['mutation_count']])
        d['mutation_count'] = mc.flatten().astype(int)
        for gene in EXPRESSION_VARS:
            d[gene] = pd.cut(df[gene], bins=[-np.inf, -0.5, 0.5, np.inf],
                             labels=[0,1,2]).astype(int)
        for v in BINARY_VARS + [OUTCOME]:
            d[v] = df[v].astype(int)
        return d
    return apply(df_tr, fit=True), apply(df_te)

tr_dom, te_dom = domain_bins(df_train, df_test)

def run_experiment(vars_list, tr_disc, te_disc):
    cols  = vars_list + [OUTCOME]
    tr_s  = to_str(tr_disc[cols])
    te_s  = to_str(te_disc[cols])
    root  = 'lymph_nodes_examined_positive' if \
            'lymph_nodes_examined_positive' in vars_list else vars_list[0]

    ts  = TreeSearch(tr_s, root_node=root)
    dag = ts.estimate(estimator_type='tan', class_node=OUTCOME, show_progress=False)
    bn  = DiscreteBayesianNetwork(dag.edges())
    for n in tr_s.columns:
        if n not in bn.nodes(): bn.add_node(n)
    bn.fit(tr_s)

    X_te = te_s.drop(columns=[OUTCOME])
    y_te = te_disc[OUTCOME].values
    p1   = bn.predict_probability(X_te)[f'{OUTCOME}_1'].values
    auc  = roc_auc_score(y_te, p1)
    f1   = f1_score(y_te, (p1 >= 0.5).astype(int))

    np.random.seed(42)
    n = len(y_te)
    boot = []
    for _ in range(1000):
        idx = np.random.choice(n, n, replace=True)
        try: boot.append(roc_auc_score(y_te[idx], p1[idx]))
        except: pass
    ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])

    return auc, f1, ci_lo, ci_hi, len(dag.edges())

# run all experiments 
EXPERIMENTS = [
    ('1. Baseline  (25 vars, quantile-3)',    ALL_VARS,    train_qnt, test_qnt),
    ('2. Clinical only (13 vars, quantile-3)',CLINICAL_VARS, train_qnt, test_qnt),
    ('3. SHAP top-10   (quantile-3)',         SHAP_TOP10,  train_qnt, test_qnt),
    ('4. Domain bins  (25 vars)',             ALL_VARS,    tr_dom,    te_dom),
    ('5. Quantile-4   (25 vars)',             ALL_VARS,    tr_q4,     te_q4),
]

print(f"{'Experiment':<42} {'Vars':>4}  {'Edges':>5}  {'AUC':>6}  "
      f"{'95% CI':>17}  {'F1':>6}  {'Time':>5}")
print("─" * 95)

exp_results = []
for name, vars_list, tr_d, te_d in EXPERIMENTS:
    t0 = time.time()
    try:
        auc, f1, lo, hi, edges = run_experiment(vars_list, tr_d, te_d)
        t = time.time() - t0
        print(f"{name:<42} {len(vars_list):>4}  {edges:>5}  {auc:.4f}  "
              f"({lo:.3f}–{hi:.3f})  {f1:.4f}  ({t:.0f}s)")
        exp_results.append({'Experiment': name, 'Vars': len(vars_list),
                            'AUC': auc, 'CI_lo': lo, 'CI_hi': hi, 'F1': f1})
    except Exception as e:
        print(f"{name:<42}  ERROR: {e}")

df_exp = pd.DataFrame(exp_results).sort_values('AUC', ascending=False)
print(f"\n{'='*50}\nBest configuration: {df_exp.iloc[0]['Experiment']}")
print(df_exp[['Experiment','Vars','AUC','CI_lo','CI_hi','F1']].to_string(index=False))


Experiment                                 Vars  Edges     AUC             95% CI      F1   Time
───────────────────────────────────────────────────────────────────────────────────────────────
1. Baseline  (25 vars, quantile-3)           25     49  0.6446  (0.589–0.698)  0.4399  (2s)
2. Clinical only (13 vars, quantile-3)       13     25  0.6411  (0.588–0.695)  0.3760  (2s)
3. SHAP top-10   (quantile-3)                10     19  0.6321  (0.579–0.688)  0.3765  (1s)
4. Domain bins  (25 vars)                    25     49  0.6556  (0.602–0.708)  0.4225  (1s)
5. Quantile-4   (25 vars)                    25     49  0.6456  (0.589–0.701)  0.4178  (1s)

Best configuration: 4. Domain bins  (25 vars)
                            Experiment  Vars      AUC    CI_lo    CI_hi       F1
             4. Domain bins  (25 vars)    25 0.655629 0.601905 0.708204 0.422535
             5. Quantile-4   (25 vars)    25 0.645633 0.588941 0.700905 0.417808
    1. Baseline  (25 vars, quantile-3)    25 0.644611 0.5

In [ ]:
# fit final models + GHC network visualization

from pgmpy.causal_discovery import HillClimbSearch as HCS, ExpertKnowledge
from pgmpy.inference import VariableElimination
BLACKLIST = [(OUTCOME, v) for v in ALL_VARS]
EK        = ExpertKnowledge(forbidden_edges=BLACKLIST)
tr_dom_str = to_str(tr_dom)

# TAN + domain bins (best predictor) → used for inference 
ts_final  = TreeSearch(tr_dom_str, root_node='lymph_nodes_examined_positive')
dag_tan   = ts_final.estimate(estimator_type='tan', class_node=OUTCOME, show_progress=False)
bn_tan    = DiscreteBayesianNetwork(dag_tan.edges())
for n in tr_dom_str.columns:
    if n not in bn_tan.nodes(): bn_tan.add_node(n)
bn_tan.fit(tr_dom_str)
print(f"TAN (domain bins): {len(dag_tan.edges())} edges  [inference model]")

# GHC + domain bins (interpretable structure) → used for visualization
hc        = HCS(scoring_method='bic-d', max_indegree=3, tabu_length=0,
                expert_knowledge=EK, return_type='dag', show_progress=False)
dag_ghc   = hc.fit(tr_dom_str).causal_graph_
bn_ghc    = DiscreteBayesianNetwork(dag_ghc.edges())
for n in tr_dom_str.columns:
    if n not in bn_ghc.nodes(): bn_ghc.add_node(n)
bn_ghc.fit(tr_dom_str)
print(f"GHC (domain bins): {len(dag_ghc.edges())} edges  [visualization model]")
print("\nEdges involving event:")
for u, v in dag_ghc.edges():
    if OUTCOME in (u, v): print(f"  {u} → {v}")

# GHC Visualization 
LABELS = {
    'lymph_nodes_examined_positive':               'Lymph Nodes+',
    '3-gene_classifier_subtype_ER+/HER2- Low Prolif': '3-Gene\nER+/HER2-',
    'type_of_breast_surgery_MASTECTOMY':           'Mastectomy',
    'neoplasm_histologic_grade':                   'Grade',
    'er_status_Positive':                          'ER+',
    'pr_status_Positive':                          'PR+',
    'her2_status_Positive':                        'HER2+',
    'age_at_diagnosis':                            'Age',
    'tumor_size':                                  'Tumor Size',
    'mutation_count':                              'Mut Count',
    'hormone_therapy':                             'Hormone Rx',
    'chemotherapy':                                'Chemo',
    'cellularity':                                 'Cellularity',
    'tp53_mut':                                    'TP53',
    'gata3_mut':                                   'GATA3',
    'pik3ca_mut':                                  'PIK3CA',
    'stat5a':                                      'STAT5A',
    'mmp15':                                       'MMP15',
    'mmp1':                                        'MMP1',
    'igf1':                                        'IGF1',
    'tbx3':                                        'TBX3',
    'hes6':                                        'HES6',
    'tubb4b':                                      'TUBB4B',
    'fancd2':                                      'FANCD2',
    'ugt2b17':                                     'UGT2B17',
    'event':                                       'DEATH',
}

G = dag_ghc

# hierarchical left-to-right layout: topological levels
from collections import defaultdict
try:
    from networkx.algorithms.dag import topological_generations
    levels = {}
    for i, gen in enumerate(topological_generations(G)):
        for n in gen: levels[n] = i
except Exception:
    levels = {n: 0 for n in G.nodes()}

by_level = defaultdict(list)
for n, l in levels.items(): by_level[l].append(n)
max_level = max(levels.values()) if levels else 1

pos = {}
for l, nodes in by_level.items():
    x = l / max_level
    for j, n in enumerate(sorted(nodes, key=lambda x: (x != OUTCOME, x))):
        y = (j - (len(nodes)-1)/2) * 0.6
        pos[n] = (x, y)

def node_color(n):
    if n == OUTCOME:            return '#E74C3C'
    if n in MUTATION_VARS:      return '#E67E00'
    if n in EXPRESSION_VARS:    return '#1A7A6E'
    return '#2980B9'

node_colors = [node_color(n) for n in G.nodes()]
labels      = {n: LABELS.get(n, n) for n in G.nodes()}

fig, ax = plt.subplots(figsize=(18, 11))
fig.patch.set_facecolor('white'); ax.set_facecolor('white')
nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=900, alpha=0.92, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels,
                        font_size=7, font_color='white',
                        font_weight='bold', ax=ax)
nx.draw_networkx_edges(G, pos, edge_color='#444444',
                       arrows=True, arrowsize=15,
                       connectionstyle='arc3,rad=0.08',
                       width=1.0, alpha=0.65, ax=ax)

import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#E74C3C', label='Outcome (death)'),
    mpatches.Patch(color='#2980B9', label='Clinical'),
    mpatches.Patch(color='#1A7A6E', label='Gene expression'),
    mpatches.Patch(color='#E67E00', label='Mutation'),
], loc='upper left', fontsize=10, framealpha=0.9)
ax.set_title('Bayesian Network (GHC) — Domain-Knowledge Discretization\n'
             f'{len(G.nodes())} nodes · {len(G.edges())} edges',
             fontsize=13, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_PATH + 'bn_ghc_structure.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("\nSaved: bn_ghc_structure.png")


TAN (domain bins): 49 edges  [inference model]
GHC (domain bins): 38 edges  [visualization model]

Edges involving event:
  lymph_nodes_examined_positive → event
  3-gene_classifier_subtype_ER+/HER2- Low Prolif → event

Saved: bn_ghc_structure.png


In [21]:
# Cell 9b (stable) — use kamada_kawai layout instead of spring

G   = dag_ghc
pos = nx.kamada_kawai_layout(G)

# Safety check — print any NaN positions
bad = [n for n, p in pos.items() if not (np.isfinite(p[0]) and np.isfinite(p[1]))]
if bad: print(f"WARNING: NaN positions for {bad}")

LABELS = {
    'lymph_nodes_examined_positive':               'Lymph\nNodes+',
    '3-gene_classifier_subtype_ER+/HER2- Low Prolif': '3-Gene\nER+/HER2-',
    'type_of_breast_surgery_MASTECTOMY':           'Mastectomy',
    'neoplasm_histologic_grade':                   'Grade',
    'er_status_Positive':                          'ER+',
    'pr_status_Positive':                          'PR+',
    'her2_status_Positive':                        'HER2+',
    'age_at_diagnosis':                            'Age',
    'tumor_size':                                  'Tumor\nSize',
    'mutation_count':                              'Mut\nCount',
    'hormone_therapy':                             'Hormone\nRx',
    'chemotherapy':                                'Chemo',
    'cellularity':                                 'Cellularity',
    'tp53_mut':                                    'TP53',
    'gata3_mut':                                   'GATA3',
    'pik3ca_mut':                                  'PIK3CA',
    'stat5a':                                      'STAT5A',
    'mmp15':                                       'MMP15',
    'mmp1':                                        'MMP1',
    'igf1':                                        'IGF1',
    'tbx3':                                        'TBX3',
    'hes6':                                        'HES6',
    'tubb4b':                                      'TUBB4B',
    'fancd2':                                      'FANCD2',
    'ugt2b17':                                     'UGT2B17',
    'event':                                       'DEATH',
}

def node_color(n):
    if n == OUTCOME:         return '#E74C3C'
    if n in MUTATION_VARS:   return '#E67E00'
    if n in EXPRESSION_VARS: return '#1A7A6E'
    return '#2980B9'

node_colors = [node_color(n) for n in G.nodes()]
labels      = {n: LABELS.get(n, n) for n in G.nodes()}
node_sizes  = [3800 if n == OUTCOME else 2400 for n in G.nodes()]

fig, ax = plt.subplots(figsize=(22, 16))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Axis limits from valid positions only
xs  = [float(p[0]) for p in pos.values() if np.isfinite(p[0])]
ys  = [float(p[1]) for p in pos.values() if np.isfinite(p[1])]
pad = 0.25
ax.set_xlim(min(xs)-pad, max(xs)+pad)
ax.set_ylim(min(ys)-pad, max(ys)+pad)

# Edges as plain lines
for u, v in G.edges():
    x0, y0 = float(pos[u][0]), float(pos[u][1])
    x1, y1 = float(pos[v][0]), float(pos[v][1])
    color  = '#C0392B' if v == OUTCOME else '#999999'
    lw     = 2.5       if v == OUTCOME else 0.9
    alpha  = 0.9       if v == OUTCOME else 0.45
    ax.plot([x0, x1], [y0, y1], '-',
            color=color, lw=lw, alpha=alpha, zorder=1)

# Nodes and labels
nx.draw_networkx_nodes(G, pos, node_color=node_colors,
                       node_size=node_sizes, alpha=0.93, ax=ax)
nx.draw_networkx_labels(G, pos, labels=labels,
                        font_size=8, font_color='white',
                        font_weight='bold', ax=ax)

import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#E74C3C', label='Outcome (death)'),
    mpatches.Patch(color='#2980B9', label='Clinical'),
    mpatches.Patch(color='#1A7A6E', label='Gene expression'),
    mpatches.Patch(color='#E67E00', label='Mutation'),
    mpatches.Patch(color='#C0392B', label='Edge → DEATH'),
], loc='upper left', fontsize=12, framealpha=0.9)

ax.set_title('Bayesian Network Structure (Hill-Climb Search)\n'
             'Domain-Knowledge Discretization  ·  '
             f'{len(G.nodes())} nodes · {len(G.edges())} edges',
             fontsize=14, fontweight='bold', pad=15)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIG_PATH + 'bn_ghc_structure.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: bn_ghc_structure.png")


Saved: bn_ghc_structure.png


In [ ]:
# inference queries (TAN + domain bins)

from pgmpy.inference import VariableElimination
ve = VariableElimination(bn_tan)

def query_death(evidence_dict):
    ev = {k: str(v) for k, v in evidence_dict.items()
          if k in bn_tan.nodes()}
    try:
        res    = ve.query([OUTCOME], evidence=ev, show_progress=False)
        states = list(res.state_names[OUTCOME])
        return float(res.values[states.index('1')])
    except Exception as e:
        print(f"  WARNING: {e}")
        return float('nan')

# Domain-bin key:
#   age:          0=<50,   1=50–65,  2=>65
#   tumor_size:   0=T1≤20, 1=T2≤50, 2=T3>50
#   lymph_nodes:  0=N0,    1=N1,     2=N2,   3=N3
#   grade:        0=G1,    1=G2,     2=G3

HIGH_RISK = dict(age_at_diagnosis=2, tumor_size=2,
                 lymph_nodes_examined_positive=2,
                 neoplasm_histologic_grade=2,
                 er_status_Positive=0, pr_status_Positive=0,
                 her2_status_Positive=0, cellularity=0,
                 chemotherapy=0, hormone_therapy=0, tp53_mut=1)

LOW_RISK  = dict(age_at_diagnosis=0, tumor_size=0,
                 lymph_nodes_examined_positive=0,
                 neoplasm_histologic_grade=0,
                 er_status_Positive=1, pr_status_Positive=1,
                 her2_status_Positive=0, cellularity=2,
                 chemotherapy=0, hormone_therapy=1, tp53_mut=0)

queries = [
    ({},                                              'Prior (no evidence)'),
    (LOW_RISK,                                        'Low-risk patient'),
    (HIGH_RISK,                                       'High-risk patient'),
    ({**HIGH_RISK, 'chemotherapy': 1},                'High-risk + chemo'),
    ({**HIGH_RISK, 'hormone_therapy': 1},             'High-risk + hormone Rx'),
    ({**HIGH_RISK, 'chemotherapy': 1,
                   'hormone_therapy': 1},             'High-risk + chemo + hormone Rx'),
    ({**LOW_RISK,  'tp53_mut': 1},                   'Low-risk + TP53 mutation'),
    ({**LOW_RISK,  'lymph_nodes_examined_positive': 2},'Low-risk + N2 lymph nodes'),
]

results = [(label, query_death(ev)) for ev, label in queries]

print(f"{'Patient profile':<42}  {'P(death=1)':>10}  {'Risk bar'}")
print("─" * 72)
for label, p in results:
    bar = '█' * int(round(p * 25))
    print(f"{label:<42}  {p:>10.4f}  {bar}")

# ── Treatment effect table ────────────────────────────────────────────────────
p_base  = query_death(HIGH_RISK)
p_chemo = query_death({**HIGH_RISK, 'chemotherapy': 1})
p_horm  = query_death({**HIGH_RISK, 'hormone_therapy': 1})
p_both  = query_death({**HIGH_RISK, 'chemotherapy': 1, 'hormone_therapy': 1})

print(f"\n── Treatment effect on high-risk patient ──")
print(f"  No treatment:      P(death) = {p_base:.4f}")
print(f"  + Chemotherapy:    P(death) = {p_chemo:.4f}   Δ = {p_chemo-p_base:+.4f}")
print(f"  + Hormone Rx:      P(death) = {p_horm:.4f}   Δ = {p_horm-p_base:+.4f}")
print(f"  + Both:            P(death) = {p_both:.4f}   Δ = {p_both-p_base:+.4f}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
labels_plot = [r[0] for r in results]
probs_plot  = [r[1] for r in results]
colors_bar  = ['#2980B9' if p < 0.4 else '#E67E00' if p < 0.6 else '#E74C3C'
               for p in probs_plot]

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(labels_plot, probs_plot, color=colors_bar, alpha=0.85, edgecolor='white')
ax.axvline(query_death({}), color='black', lw=1.5, ls='--', alpha=0.6,
           label=f'Prior = {query_death({}):.3f}')
for bar, p in zip(bars, probs_plot):
    ax.text(p + 0.005, bar.get_y() + bar.get_height()/2,
            f'{p:.3f}', va='center', fontsize=10)
ax.set_xlim(0, 1)
ax.set_xlabel('P(death = 1)', fontsize=12)
ax.set_title('Bayesian Network Inference — P(death | patient profile)\n'
             'Model: TAN + domain-knowledge discretization', fontsize=12)
ax.legend(fontsize=10)
ax.invert_yaxis()
fig.patch.set_facecolor('white')
ax.set_facecolor('#F8F8F8')
plt.tight_layout()
plt.savefig(FIG_PATH + 'bn_inference_queries.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("\nSaved: bn_inference_queries.png")


Patient profile                             P(death=1)  Risk bar
────────────────────────────────────────────────────────────────────────
Prior (no evidence)                             0.3268  ████████
Low-risk patient                                0.0493  █
High-risk patient                               0.4655  ████████████
High-risk + chemo                               0.7878  ████████████████████
High-risk + hormone Rx                          0.5357  █████████████
High-risk + chemo + hormone Rx                  0.8310  █████████████████████
Low-risk + TP53 mutation                        0.1061  ███
Low-risk + N2 lymph nodes                       0.1331  ███

── Treatment effect on high-risk patient ──
  No treatment:      P(death) = 0.4655
  + Chemotherapy:    P(death) = 0.7878   Δ = +0.3223
  + Hormone Rx:      P(death) = 0.5357   Δ = +0.0702
  + Both:            P(death) = 0.8310   Δ = +0.3655

Saved: bn_inference_queries.png


In [ ]:
#cCell 11 — Experiment comparison figure (forest plot + table)

fig, (ax_plot, ax_table) = plt.subplots(
    2, 1, figsize=(13, 8),
    gridspec_kw={'height_ratios': [3, 1.2]})
fig.patch.set_facecolor('white')

# ── Data ─────────────────────────────────────────────────────────────────────
exp_names = [
    '5. Quantile-4\n(25 vars)',
    '4. Domain bins\n(25 vars) ★',
    '1. Baseline\n(25 vars, quantile-3)',
    '2. Clinical only\n(13 vars, quantile-3)',
    '3. SHAP top-10\n(quantile-3)',
]
aucs   = [0.6456, 0.6556, 0.6446, 0.6411, 0.6321]
ci_lo  = [0.5889, 0.6019, 0.5887, 0.5882, 0.5786]
ci_hi  = [0.7009, 0.7082, 0.6976, 0.6953, 0.6883]
f1s    = [0.4178, 0.4225, 0.4399, 0.3760, 0.3765]
colors = ['#E74C3C' if i == 1 else '#2980B9' for i in range(5)]

# Sort bottom-to-top for horizontal plot
order = list(range(4, -1, -1))
names_s = [exp_names[i] for i in order]
aucs_s  = [aucs[i]  for i in order]
lo_err  = [aucs[i] - ci_lo[i] for i in order]
hi_err  = [ci_hi[i] - aucs[i] for i in order]
cols_s  = [colors[i] for i in order]

# ── Forest plot ───────────────────────────────────────────────────────────────
y_pos = range(len(names_s))
ax_plot.set_facecolor('white')

for y, auc, lo, hi, col in zip(y_pos, aucs_s, lo_err, hi_err, cols_s):
    ax_plot.errorbar(auc, y, xerr=[[lo], [hi]],
                     fmt='o', color=col, markersize=9,
                     elinewidth=2, capsize=5, capthick=2, zorder=3)
    ax_plot.axhline(y, color='#DDDDDD', lw=0.8, zorder=1)

ax_plot.axvline(0.5, color='black', lw=1.0, ls=':', alpha=0.4)

# Shade best model
best_y = order.index(1)
ax_plot.axhspan(best_y - 0.4, best_y + 0.4,
                color='#FADBD8', alpha=0.5, zorder=0)

ax_plot.set_yticks(list(y_pos))
ax_plot.set_yticklabels(names_s, fontsize=10)
ax_plot.set_xlabel('AUC (95% Bootstrap CI)', fontsize=11)
ax_plot.set_xlim(0.50, 0.78)
ax_plot.set_title('Bayesian Network — Configuration Comparison (Test Set AUC)\n'
                  'Model: TAN + 5-fold CV selection',
                  fontsize=12, fontweight='bold')
ax_plot.spines[['top','right']].set_visible(False)

ax_table.axis('off')
table_data = [
    ['Configuration',      'Vars', 'AUC',   '95% CI',           'F1'],
    ['Domain bins ★',      '25',   '0.656', '0.602 – 0.708',    '0.423'],
    ['Quantile-4',         '25',   '0.646', '0.589 – 0.701',    '0.418'],
    ['Quantile-3 baseline','25',   '0.645', '0.589 – 0.698',    '0.440'],
    ['Clinical only',      '13',   '0.641', '0.588 – 0.695',    '0.376'],
    ['SHAP top-10',        '10',   '0.632', '0.579 – 0.688',    '0.377'],
]

tbl = ax_table.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc='center', loc='center',
    bbox=[0, 0, 1, 1]
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)

# Style header
for j in range(5):
    tbl[(0, j)].set_facecolor('#2C3E50')
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')
# Style best row
for j in range(5):
    tbl[(1, j)].set_facecolor('#FADBD8')
    tbl[(1, j)].set_text_props(fontweight='bold')

plt.tight_layout(h_pad=2)
plt.savefig(FIG_PATH + 'bn_experiment_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.close()
print("Saved: bn_experiment_comparison.png")


Saved: bn_experiment_comparison.png
